In [1]:
print(1)

1


# Step 3

In [2]:
import json
import shutil
from pathlib import Path

import anndata as ad
import numpy as np
import os
import pandas as pd

In [3]:
from tqdm import tqdm

In [4]:
import pandas as pd

In [5]:
SHARDSIZE = 200_000

In [6]:
PLIBDATA_ROOT = '/home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets'
SHARDS_ROOT = Path('/home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata')

JOIN_KEYS = ['dataset', 'context', 'perturbation']
SHARD_COLS = ['dataset', 'context', 'perturbation', 'readout', 'log_dose', 'time', 'value', 'split']

In [8]:
df_annot_split = pd.read_parquet('/home/icb/olga.novitskaia/lpm_style/.plib_cache/annotations/df_annot_split.parquet')

In [10]:
def _append_context_chunk(
    ctx_df: pd.DataFrame,
    out_dir: Path,
    shardsize: int,
    next_shard_id: int,
) -> tuple[int, list[dict]]:
    """Append one chunk of rows to a (dataset, context) folder as plib-style shards.

    The folder is NOT wiped here — call `_reset_context_folder(out_dir)` once before the first
    chunk if you want a clean slate. Within a chunk, every emitted shard has a single split
    and at most `shardsize` rows. Shards from different chunks may belong to the same split,
    so a folder can contain several smaller-than-`shardsize` "tail" shards.

    Returns (next_shard_id_after_chunk, metadata_rows_for_chunk).
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    long = ctx_df[SHARD_COLS].copy()
    long['perturbation'] = long['perturbation'].astype(str)
    long = long.sort_values('split', kind='mergesort').reset_index(drop=True)
    long['shard_id_per_split'] = long.groupby('split').cumcount() // shardsize
    combo = long['split'].astype(str) + '::' + long['shard_id_per_split'].astype(str)
    long['shard_id'] = pd.factorize(combo)[0] + next_shard_id
    long = long.drop(columns='shard_id_per_split')

    metadata_rows = []
    max_id = next_shard_id - 1
    for shard_id, shard_df in long.groupby('shard_id', sort=True):
        shard_name = f'shard_{int(shard_id):06d}.parquet'
        shard_df.drop(columns=['split', 'shard_id']).reset_index(drop=True).to_parquet(
            out_dir / shard_name, engine='pyarrow', index=False,
        )
        metadata_rows.append({
            'shard_path': f'{out_dir.name}/{shard_name}',
            'size': int(len(shard_df)),
            'split': str(shard_df['split'].iloc[0]),
            'context': str(shard_df['context'].iloc[0]),
            'datasets': str(shard_df['dataset'].iloc[0]),
            'perturbations': sorted({p for s in shard_df['perturbation'] for p in s.split('+')}),
            'log_doses': sorted(shard_df['log_dose'].dropna().unique().tolist()),
            'times': sorted(shard_df['time'].dropna().unique().tolist()),
            'readouts': shard_df['readout'].unique().tolist(),
        })
        max_id = max(max_id, int(shard_id))

    return max_id + 1, metadata_rows


def _reset_context_folder(out_dir: Path, shardsize: int) -> None:
    """Wipe a (dataset, context) folder and write info.json immediately so the folder is
    self-describing even before the first metadata.parquet is finalized."""
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in out_dir.iterdir():
        (shutil.rmtree if f.is_dir() else Path.unlink)(f)
    (out_dir / 'info.json').write_text(json.dumps({
        'PDATA_FORMAT_VERSION': 1,
        'CONTEXT_MODULE_HASH': '',
        'SHARDSIZE': shardsize,
    }))


def _finalize_context_folder(out_dir: Path, metadata_rows: list[dict]) -> None:
    """Write metadata.parquet for a folder once all chunks have been appended."""
    pd.DataFrame(metadata_rows).to_parquet(out_dir / 'metadata.parquet', engine='pyarrow', index=False)

In [11]:
from collections import defaultdict

In [12]:
SHARDS_ROOT.mkdir(parents=True, exist_ok=True)
print(f'shards will be written under: {SHARDS_ROOT}')
print(f'source datasets in {PLIBDATA_ROOT}: {sorted(os.listdir(PLIBDATA_ROOT))}')

shards will be written under: /home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata
source datasets in /home/icb/olga.novitskaia/lpm_style/.plib_cache/raw_datasets: ['cigs_mce', 'cigs_tcm', 'dili_train', 'gdpx2', 'l1000_phase1', 'l1000_phase2', 'novartis', 'tahoe', 'vcpi_0001', 'vcpi_0002']


In [13]:
# Drive the per-(dataset, context) sharding from df_annot_split, processing one source parquet
# file at a time. Folder layout mirrors perturblib's plibdata: a single FLAT folder per
# (dataset, context) named `{dataset}_{context}`, sitting directly under SHARDS_ROOT. Each
# folder contains info.json + metadata.parquet + shard_NNNNNN.parquet files. We never concat
# a whole dataset into memory; each source parquet is loaded, joined with the split lookup,
# grouped by context, and *appended* to its target folder. A folder is wiped exactly once --
# the first time any chunk lands in it during this run -- so re-running the cell is idempotent.

def _folder_name(dataset_folder: str, context: str) -> str:
    """Flat per-(dataset, context) folder name, perturblib-style."""
    return f'{dataset_folder}_{context}'

# split lookup keyed on the natural identity columns of a DEG row
split_lookup = df_annot_split[JOIN_KEYS + ['split']].drop_duplicates(JOIN_KEYS)

source_folders = sorted(p for p in os.listdir(PLIBDATA_ROOT) if (Path(PLIBDATA_ROOT) / p).is_dir())

shard_counters: dict[str, int] = defaultdict(int)
metadata_per_folder: dict[str, list[dict]] = defaultdict(list)
folder_initialized: set[str] = set()

for dataset_folder in tqdm(source_folders, desc='datasets'):
    dataset_dir = Path(PLIBDATA_ROOT) / dataset_folder
    files = sorted(f for f in os.listdir(dataset_dir) if f.endswith('.parquet'))
    if not files:
        print(f'  [skip] {dataset_folder}: no parquet files')
        continue

    touched_in_this_dataset: set[str] = set()

    for file in tqdm(files, desc=dataset_folder, leave=False):
        df = pd.read_parquet(dataset_dir / file)
        if df.empty:
            continue

        n_before = len(df)
        df = df.merge(split_lookup, on=JOIN_KEYS, how='inner')
        n_lost = n_before - len(df)
        if n_lost:
            print(f'  [warn] {dataset_folder}/{file}: {n_lost} rows had no split assignment, dropped')
        if df.empty:
            continue

        for context, ctx_df in df.groupby('context', sort=True):
            folder_name = _folder_name(dataset_folder, str(context))
            out_dir = SHARDS_ROOT / folder_name

            # first time we touch this folder in the current run: wipe + write info.json
            if folder_name not in folder_initialized:
                _reset_context_folder(out_dir, SHARDSIZE)
                folder_initialized.add(folder_name)

            next_id, meta = _append_context_chunk(
                ctx_df, out_dir, SHARDSIZE, shard_counters[folder_name],
            )
            shard_counters[folder_name] = next_id
            metadata_per_folder[folder_name].extend(meta)
            touched_in_this_dataset.add(folder_name)

    # finalize metadata.parquet for every folder this dataset touched, so per-dataset
    # progress is durable even if a later dataset fails or is interrupted.
    for folder_name in touched_in_this_dataset:
        _finalize_context_folder(SHARDS_ROOT / folder_name, metadata_per_folder[folder_name])

total_shards = sum(len(v) for v in metadata_per_folder.values())
total_folders = len(metadata_per_folder)
print(f'done: wrote {total_shards} shards across {total_folders} dataset_context folders -> {SHARDS_ROOT}')

datasets: 100%|██████████| 10/10 [2:00:15<00:00, 721.53s/it] 

done: wrote 11760 shards across 160 dataset_context folders -> /home/icb/olga.novitskaia/lpm_style/.plib_cache/plibdata
